In [1]:
from pension import Liabilities
from portfolio import *
from cma import *
from math import sqrt
from pprint import pprint

## Representing Liabilities

The `Liabilities` object represents pension liabilities based on our capital markets assumptions. This Projects one path of liability growth over any horizon.

In [2]:
l = Liabilities(
    retired_members=RETIRED_MEMBERS,
    active_members=ACTIVE_MEMBERS,
    average_salary=AVG_SALARY,
    min_active_members=MIN_ACTIVE_MEMBERS,
    active_members_decline=ACTIVE_MEMBER_DECLINE,
    retired_members_growth=RETIRED_MEMBERS_GROWTH,
    wage_growth_rate=WAGE_GROWTH,
    starting_duration=INITIAL_DURATION,
    actuarial_df=ACTUARIAL_DF,
    service_cost=SERVICE_COST_RATE,
    starting_liabilities=LIABILITIES,
    starting_benefit=STARTING_BENEFIT,
    benefit_growth_rate=BENEFIT_GROWTH_RATE,
    liabilities_cache={}
)


In [3]:
l.get_closing_liabilities(30)

9407397613.929964

In [4]:
l.benefits_paid(1)

83200000.0

In [5]:
l.get_interest_cost(1)

63600000.0

In [6]:
l.get_service_cost(1)

129364725.0

## Asset Configurations
We have preset Asset configurations for the 9 core classes we currently hold in the portfolio (can easily be expanded with new asset classes). Each contains an expected return, variance, and liquidity rating.

In [7]:
cash = Cash()
can_equity = CanEquity()
us_equity = USEquity()
em_equity = EMEquity()
id_equity = IDEquity()
fixed_income = FixedIncome()
private_equity = PrivateEquity()
infrastructure = Infrastructure()
real_estate = RealEstate()

## Asset Allocation 
The `AssetAlloc` object allows us to build multiple portfolios for testing. You simply create your assets (as above) and assign weights to them.

In [21]:
a = AssetAlloc(
    [
        (cash, 0.02),
        (can_equity, 0.1),
        (us_equity, 0.09),
        (em_equity, 0.05),
        (id_equity, 0.04),
        (fixed_income, 0.28),
        (private_equity, 0.13),
        (infrastructure, 0.13),
        (real_estate, 0.16),

    ]
)

In [9]:
a.get_expected_return()

0.061090000000000005

In [10]:
a.get_std_dev()

0.07771579057847504

In [11]:
a.get_sharpe()

0.4000474005164532

## Portfolio
The `Portfolio` object combines the `Liabilities` projection with the `AssetAlloc`. These two components make up the full pension portfolio. This runs background simulations to give us estimates of our portfolio's performance. 

In [12]:
p = Portfolio(a, l)

In [13]:
p.get_var(ci=0.95, scenario="base", horizon=1)

67134628.80722067

In [14]:
float(p.funded_ratio_vol(horizon=30, scenario="base"))

0.4880054764254675

In [15]:
p.get_cvar(ci=0.95, scenario="base")

92458926.6019381

In [16]:
p.underfunding_probability()

0.7909

In [17]:
float(p.portfolio_avg())

3130409422.6034923

In [18]:
float(p.funded_ratio_avg())

0.7739507016012278

## Scenarios
To model scenarios, the `Portfolio` object loads in with base scenarios:
* base
* bull
* bear
* stagflation
* gfc

But we also have the ability to create new scenarios

In [19]:
my_scenario = {
    "returns" : {
        "cash" : 0.0,
        "fixed_income" : 0.01,
        "can_equity" : -0.03,
        "us_equity" : -0.05,
        "id_equity" : -0.03,
        "em_equity" : -0.02,
        "real_estate" : 0.0,
        "infrastructure" : 0.0,
        "private_equity" : -0.06,
    } 
}
p.add_scenario("test_scenario_1", my_scenario)

In [20]:
p.get_var(scenario="test_scenario_1")

77623965.65519401

## Discussion Points

### Monte Carlo Methodology

The excel model uses a simple random return generation method that uses the portfolio's expected returns and variance to generate on return path for simulation. 

The current python approach uses brownian motion to project asset paths individually and then correlate them back to eachother.

These yield very different results:

Excel P(underfunding) ~ 0.15

Python  P(underfunding) ~ 0.8

The Brownian motion approach is industry standard but may not be necessary for our purposes. It assumes continuous price movements and constant returns + volatility. There are more sophisticated approaches to our portfolio setup that might be better. We may want to model different asset classes with different distributions

I have an early implementation of the Excel version, but want to consult before moving forward with that. This one is simpler and easier to understand + maintain
